# Train the jersey number ResNet on Colab

Trains `jersey_net.py`'s ResNet on your jersey dataset and saves `jersey_model.pt` to Google Drive.

**Before running:**
1. *Runtime → Change runtime type → GPU* (a T4 is fine).
2. In Google Drive, make a folder `Hockey-Vision` (or change `DRIVE_DIR` below) containing:
   - `jersey_net.py` (repo root), `make_jersey_labels.py` and `train_jersey.py` (repo `colab/` folder)
   - your dataset, either zipped as `jersey_dataset.zip` (a Roboflow *YOLOv8* export, with its
     `data.yaml`), **or** fill in the `ROBOFLOW_*` options to download it from Roboflow directly:
     add your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left
     sidebar) and give this notebook access to it
3. Set the options in the next cell, then *Runtime → Run all*.

The best epoch is saved to Drive whenever it improves, so if Colab disconnects you keep the best
model so far. Copy `jersey_model.pt` into `CV_Models/jersey_Num Models/` in the repo afterwards.

**Roboflow export settings:** export the version *without* flip augmentation (it mirrors the
digits) and preferably without any augmentation (training augments on its own). *Stretch* resizing
is fine; *Fit (black edges)* letterboxing is not, since it shifts where the torso is in the image.

In [ ]:
# ---- Options ----
DRIVE_DIR = "/content/drive/MyDrive/Hockey-Vision"   # folder with the .py files and the dataset zip
DATASET_ZIP = "jersey_dataset.zip"                   # inside DRIVE_DIR, if not downloading from Roboflow
# To download from Roboflow instead, fill these in (from the project's URL:
# app.roboflow.com/<workspace>/<project>/<version>) and add the ROBOFLOW_API_KEY secret.
ROBOFLOW_WORKSPACE = ""
ROBOFLOW_PROJECT = ""
ROBOFLOW_VERSION = 0
IMAGES_ARE_TORSO = False   # True if the dataset images are already torso crops, not whole players
ARCH = "resnet34"          # "resnet18" (lighter), "resnet34", or "resnet50" (the larger model)
INPUT_HW = (160, 128)      # crop height, width; (224, 176) for more detail (taller than wide, multiples of 16)
EPOCHS = 30
BATCH = 128                # 64 for resnet50 at (224, 176) if the T4 runs out of memory
OUTPUT_NAME = f"jersey_model_{ARCH}_{INPUT_HW[0]}x{INPUT_HW[1]}.pt"              # saved in DRIVE_DIR

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU, then rerun"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Get the code and dataset

The dataset goes onto the Colab machine's own disk: reading thousands of small images straight
from Drive is very slow.

In [ ]:
import shutil, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
drive_dir = Path(DRIVE_DIR)
code_dir = Path("/content/hv")
code_dir.mkdir(exist_ok=True)
for name in ("jersey_net.py", "make_jersey_labels.py", "train_jersey.py"):
    if not (drive_dir / name).exists():
        raise FileNotFoundError(f"Put {name} from the repo in {drive_dir}")
    shutil.copy(drive_dir / name, code_dir / name)
sys.path.insert(0, str(code_dir))

raw_dir = Path("/content/dataset")
if not raw_dir.exists():
    if ROBOFLOW_PROJECT:
        !pip install -q roboflow
        from google.colab import userdata
        from roboflow import Roboflow
        rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
        version = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION)
        version.download("yolov8", location=str(raw_dir))
    else:
        with zipfile.ZipFile(drive_dir / DATASET_ZIP) as z:
            z.extractall(raw_dir)
yamls = sorted(raw_dir.rglob("data.yaml"), key=lambda p: len(p.parts))
if not yamls:
    raise FileNotFoundError(f"No data.yaml in {DATASET_ZIP}")
DATASET_DIR = yamls[0].parent
print("Dataset:", DATASET_DIR)

## 3. Make whole-number labels

Check the summary: there should be `train` and `valid` splits, a decent share of crops with no
number (roughly 20-40%), and few skipped images. Take the warnings seriously: Roboflow splits
images at random, so frames of the same clip often end up in both train and valid, which makes
validation accuracy look better than it is. If it warns, re-split by game in Roboflow (or move
whole clips between the split folders) before trusting the numbers.

In [ ]:
torso_flag = "--torso" if IMAGES_ARE_TORSO else ""
!cd /content/hv && python make_jersey_labels.py "{DATASET_DIR}" {torso_flag}

## 4. Pre-cut the training crops

Colab has only a couple of CPU cores, so decoding and cropping full-size images each epoch would
leave the GPU waiting. This cuts every image to exactly what the model sees (the torso band,
resized to `INPUT_HW`) once, as lossless PNGs. Training on these with `--torso` is
identical to training on the originals, just faster.

In [ ]:
import csv, cv2
from concurrent.futures import ThreadPoolExecutor
from jersey_net import prepare, torso_crop

PREP_DIR = Path(f"/content/prepped_{INPUT_HW[0]}x{INPUT_HW[1]}")  # one folder per input size
rows = list(csv.DictReader(open(DATASET_DIR / "labels.csv")))

def cut(row):
    img = cv2.imread(str(DATASET_DIR / row["path"]))
    if img is None:
        return None
    if not IMAGES_ARE_TORSO:
        img = torso_crop(img, (0, 0, img.shape[1], img.shape[0]))
    if img.shape[0] < 8 or img.shape[1] < 8:
        return None
    out = PREP_DIR / Path(row["path"]).with_suffix(".png")
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out), cv2.cvtColor(prepare(img, INPUT_HW), cv2.COLOR_RGB2BGR))
    return {**row, "path": out.relative_to(PREP_DIR).as_posix()}

with ThreadPoolExecutor(8) as pool:
    prepped = [r for r in pool.map(cut, rows) if r is not None]
with open(PREP_DIR / "labels.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["path", "number", "split"])
    w.writeheader()
    w.writerows(prepped)
print(f"{len(prepped)} of {len(rows)} images cut into {PREP_DIR}")

## 5. Look at some crops

These are exactly what the model trains on. The number should be clearly inside each crop; if
it's often cut off at the top or bottom, `TORSO_Y` in `jersey_net.py` needs changing.

In [ ]:
import random
import matplotlib.pyplot as plt

sample = random.sample([r for r in prepped if r["split"] == "train"], min(32, len(prepped)))
fig, axes = plt.subplots(4, 8, figsize=(16, 10))
for ax, r in zip(axes.flat, sample):
    ax.imshow(cv2.cvtColor(cv2.imread(str(PREP_DIR / r["path"])), cv2.COLOR_BGR2RGB))
    ax.set_title(r["number"] or "none")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Train

`numbered` is the accuracy on crops that show a number, the one to watch. The best epoch
(`balanced`) is saved to Drive as it improves. The final lines list the most common mistakes.

In [ ]:
import os
output = drive_dir / OUTPUT_NAME
workers = os.cpu_count()
!cd /content/hv && python train_jersey.py "{PREP_DIR}" --torso --arch {ARCH} --input-hw {INPUT_HW[0]} {INPUT_HW[1]} --epochs {EPOCHS} --batch {BATCH} --workers {workers} --output "{output}"

## 7. Done

The model is in Drive at `DRIVE_DIR/OUTPUT_NAME`. Put it in the repo's `CV_Models/jersey_Num Models/` (the name can stay as it is),
copy that to the Jetson, and there run:

```bash
python export_engines.py --models jersey
jetson/run_clip.sh videos/<clip>.mp4 --number-reader resnet
```

To also download it straight to your computer, run the cell below.

In [ ]:
from google.colab import files
files.download(str(drive_dir / OUTPUT_NAME))